In [1]:

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix,

)

In [2]:

DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

df_books[["상품명", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설
5,초역 부처의 말,인문
6,청춘의 독서(특별증보판),인문
7,어른의 행복은 조용하다,시/에세이
8,채식주의자,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),시/에세이


In [3]:
df_model = df_books[["상품명", "분야"]].copy()

for col in ["상품명", "분야"]:

    df_model[col] = (

        df_model[col]

        .fillna("")

        .astype(str)

        .str.strip()

    )

df_model = df_model[

    (df_model["상품명"] != "") &

    (df_model["분야"] != "")

].reset_index(drop=True)

print("모델링 데이터 크기:", df_model.shape)

모델링 데이터 크기: (199, 2)


In [4]:

print("분야 종류 수:", df_model["분야"].nunique())

df_model["분야"].value_counts().head(20)


class_counts = df_model["분야"].value_counts()

class_counts[class_counts < 2]

분야 종류 수: 16


분야
예술/대중문화    1
컴퓨터/IT     1
Name: count, dtype: int64

In [6]:
# 입력 데이터(X)와 정답 데이터(y) 정의
X = df_model["상품명"]
y = df_model["분야"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

print("X 예시:")
print(X.head())

print("\ny 예시:")
print(y.head())

X 크기: (199,)
y 크기: (199,)
X 예시:
0        소년이 온다
1            모순
2    결국 국민이 합니다
3           혼모노
4            급류
Name: 상품명, dtype: str

y 예시:
0       소설
1       소설
2    정치/사회
3       소설
4       소설
Name: 분야, dtype: str


In [9]:
# 분야별 데이터 개수 확인
category_counts = df_model["분야"].value_counts()

# 데이터가 2개 이상인 분야만 선택
valid_categories = category_counts[category_counts >= 2].index

df_model = df_model[
    df_model["분야"].isin(valid_categories)
].copy()

# 다시 X, y 정의
X = df_model["상품명"]
y = df_model["분야"]

print(y.value_counts())

분야
소설         48
인문         27
경제/경영      26
시/에세이      25
자기계발       17
외국어        15
어린이(초등)    10
정치/사회       7
청소년         5
과학          5
역사/문화       5
가정/육아       3
요리          2
만화          2
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터:", len(X_train))
print("테스트 데이터:", len(X_test))

학습 데이터: 157
테스트 데이터: 40


In [11]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF:", X_train_tfidf.shape)

print("Test TF-IDF :", X_test_tfidf.shape)

Train TF-IDF: (157, 447)
Test TF-IDF : (40, 447)


In [12]:

model = MultinomialNB()

In [13]:

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

In [14]:

result = pd.DataFrame({

    "상품명": X_test.reset_index(drop=True),

    "실제_분야": y_test.reset_index(drop=True),

    "예측_분야": y_pred,

})

result.head(20)

,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
1,흰,소설,소설
2,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
3,최소한의 한국사,역사/문화,소설
4,왜 그들만 부자가 되는가,경제/경영,소설
5,죽이고 싶은 아이,청소년,소설
6,침묵의 퍼레이드,소설,소설
7,행동은 불안을 이긴다,자기계발,소설
8,더 나은 어휘를 쓰고 싶은 당신을 위한 필사책,인문,인문
9,내가 한 말을 내가 오해하지 않기로 함,시/에세이,소설


In [15]:

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.3500


In [16]:
report = classification_report(

    y_test,

    y_pred,

    zero_division=0,

)

print(report)

              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         1
       경제/경영       1.00      0.20      0.33         5
          과학       0.00      0.00      0.00         1
          소설       0.28      1.00      0.43        10
       시/에세이       0.00      0.00      0.00         5
     어린이(초등)       0.00      0.00      0.00         2
       역사/문화       0.00      0.00      0.00         1
         외국어       1.00      0.33      0.50         3
          인문       1.00      0.33      0.50         6
        자기계발       0.00      0.00      0.00         4
       정치/사회       0.00      0.00      0.00         1
         청소년       0.00      0.00      0.00         1

    accuracy                           0.35        40
   macro avg       0.27      0.16      0.15        40
weighted avg       0.42      0.35      0.26        40



In [17]:
labels = model.classes_

cm = confusion_matrix(

    y_test,

    y_pred,

    labels=labels,

)

df_cm = pd.DataFrame(

    cm,

    index=labels,

    columns=labels,

)

df_cm

,가정/육아,경제/경영,과학,만화,소설,시/에세이,어린이(초등),역사/문화,외국어,요리,인문,자기계발,정치/사회,청소년
가정/육아,0,0,0,0,1,0,0,0,0,0,0,0,0,0
경제/경영,0,1,0,0,4,0,0,0,0,0,0,0,0,0
과학,0,0,0,0,1,0,0,0,0,0,0,0,0,0
만화,0,0,0,0,0,0,0,0,0,0,0,0,0,0
소설,0,0,0,0,10,0,0,0,0,0,0,0,0,0
시/에세이,0,0,0,0,5,0,0,0,0,0,0,0,0,0
어린이(초등),0,0,0,0,2,0,0,0,0,0,0,0,0,0
역사/문화,0,0,0,0,1,0,0,0,0,0,0,0,0,0
외국어,0,0,0,0,2,0,0,0,1,0,0,0,0,0
요리,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:

misclassified = result[

    result["실제_분야"] != result["예측_분야"]

].copy()

print("오분류 수:", len(misclassified))

misclassified.head(20)

오분류 수: 26


,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
2,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
3,최소한의 한국사,역사/문화,소설
4,왜 그들만 부자가 되는가,경제/경영,소설
5,죽이고 싶은 아이,청소년,소설
7,행동은 불안을 이긴다,자기계발,소설
9,내가 한 말을 내가 오해하지 않기로 함,시/에세이,소설
11,쇼펜하우어 인생수업(30만 부 기념 개정증보판),인문,소설
13,나는 메트로폴리탄 미술관의 경비원입니다,시/에세이,소설
14,단 3개의 미국 ETF로 은퇴하라,경제/경영,소설


In [19]:

result.to_csv(

    "chapter04_predictions.csv",

    index=False,

    encoding="utf-8-sig",

)

misclassified.to_csv(

    "chapter04_misclassified.csv",

    index=False,

    encoding="utf-8-sig",

)

In [20]:

new_titles = [

    "파이썬으로 시작하는 데이터 분석",

    "처음 배우는 주식 투자",

    "마음을 이해하는 심리학",

]

new_vectors = tfidf.transform(new_titles)

new_predictions = model.predict(new_vectors)

pd.DataFrame({

    "상품명": new_titles,

    "예상_분야": new_predictions,

})

,상품명,예상_분야
0,파이썬으로 시작하는 데이터 분석,소설
1,처음 배우는 주식 투자,소설
2,마음을 이해하는 심리학,소설
